# Notebook 5: Text Feature Engineering

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 60 min

**Week 12 Theme:** Airbnb-style listing price prediction

## Learning Objectives
By the end of this notebook you will be able to:
- Build `CountVectorizer` and `TF-IDF` from scratch and understand every step
- Explain what n-grams are and when bigrams help
- Extract text features from Airbnb listing descriptions
- Combine text features with numeric features inside a single pipeline


## Analogy: TF-IDF as a Document Fingerprint

A TF-IDF vector is like a **fingerprint of a document**.

- The word **"cozy"** appears in almost every Airbnb listing. It's like a fingerprint smudge everyone shares — it tells you nothing distinctive.
- The phrase **"chef's kitchen"** appears in only 5% of listings. That's a rare fingerprint marker — if a listing has it, you immediately know something meaningful about it.

**TF-IDF rewards words that are:**
1. **Frequent in a specific document** (Term Frequency — the word really describes *this* listing)
2. **Rare across all documents** (Inverse Document Frequency — it's distinctive, not generic)

A word that is both common in one listing and rare across all listings gets a **high TF-IDF score** — it's the listing's unique fingerprint marker.


## Why Does This Matter for ML?

Text appears everywhere in real-world ML systems:
- **Product descriptions** (e-commerce, Airbnb, job postings)
- **Customer reviews** — sentiment, quality signals
- **Support tickets** — urgency, topic classification
- **Social media** — intent detection, trend analysis

Text features often capture **intent and quality** that numeric features miss entirely. A listing with 3 bedrooms and a 4.8-star rating tells you quantity. The phrase "recently renovated, floor-to-ceiling windows" tells you *why* it commands a premium.

**Rule of thumb:** In any domain with text, ignoring it typically leaves 10–30% model performance on the table.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import scipy.sparse

np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Synthetic Airbnb listing descriptions ──────────────────────────────────
# Four listing archetypes, each with slight random variation
cozy_templates = [
    "Cozy studio near the centre, perfect for solo travellers",
    "Warm and cozy room close to the city centre",
    "Charming cozy apartment just steps from the high street",
    "Snug and cozy studio with great transport links",
    "Cozy private room in a shared flat near the centre",
]

luxury_templates = [
    "Luxury apartment with chef kitchen and panoramic views",
    "Stunning luxury penthouse featuring panoramic city views and chef kitchen",
    "High-end luxury flat with designer furnishings and panoramic views",
    "Exclusive luxury apartment, chef kitchen, rooftop terrace with panoramic views",
    "Premium luxury suite with chef kitchen, panoramic views and concierge service",
]

budget_templates = [
    "Budget-friendly room perfect for backpackers on a tight budget",
    "Affordable budget room, shared bathroom, great for backpackers",
    "Clean basic room ideal for budget backpackers",
    "Simple budget accommodation for backpackers and solo travellers",
    "No-frills budget room, shared kitchen, perfect for backpackers",
]

family_templates = [
    "Spacious family home with garden, perfect for families with children",
    "Large family house, big garden, child-friendly neighbourhood",
    "Comfortable family property with enclosed garden and play area for children",
    "Family-friendly house, spacious garden, close to schools and parks",
    "Welcoming family home with large garden and safe neighbourhood for children",
]

business_templates = [
    "Modern business apartment with high-speed WiFi and dedicated workspace",
    "Professional business flat, fast WiFi, private workspace near financial district",
    "Corporate business suite with ergonomic workspace and gigabit WiFi",
    "Executive business apartment, quiet workspace, high-speed WiFi included",
    "Smart business studio with dedicated workspace and ultrafast WiFi",
]

# Build 200 descriptions: 40 per type (cycle through templates)
all_templates = [
    (cozy_templates,   'cozy',     50),   # base price ~80
    (luxury_templates, 'luxury',  150),   # base price ~250
    (budget_templates, 'budget',   25),   # base price ~40
    (family_templates, 'family',  100),   # base price ~160
    # We'll use 4 types, 50 each = 200 total
]

descriptions, listing_types, prices = [], [], []
np.random.seed(42)

type_configs = [
    (cozy_templates,    'cozy',    80,  20),
    (luxury_templates,  'luxury',  250, 60),
    (budget_templates,  'budget',  40,  10),
    (family_templates,  'family',  160, 40),
]

for templates, ltype, base_price, price_std in type_configs:
    for i in range(50):                                   # 50 listings per type
        desc = templates[i % len(templates)]              # cycle through templates
        descriptions.append(desc)
        listing_types.append(ltype)
        price = max(10, base_price + np.random.randn() * price_std)  # add noise
        prices.append(round(price, 2))

df = pd.DataFrame({
    'description': descriptions,
    'listing_type': listing_types,
    'price': prices,
    'bedrooms': np.random.randint(1, 5, 200),
    'review_score': np.clip(np.random.normal(4.3, 0.4, 200), 1, 5).round(1),
})

print(f"Dataset shape: {df.shape}")
print(f"Listing type counts:\n{df['listing_type'].value_counts()}")
df.head()

## Bag of Words (BoW): Text as a Vector of Word Counts

The simplest text representation: count how many times each word appears in a document.

**Example:**
```
"Cozy studio near centre"  →  {cozy:1, studio:1, near:1, centre:1}
```

**What you gain:** Word presence and frequency are captured.  
**What you lose:** Word order — "dog bites man" and "man bites dog" produce identical BoW vectors.

Despite this limitation, BoW is surprisingly powerful for classification and regression tasks where the *presence* of words matters more than their sequence.

The result is a **sparse matrix**: most cells are 0 because any given document contains only a small fraction of all possible words.


In [ ]:
np.random.seed(42)

class CountVectorizerScratch:
    """Bag-of-Words vectorizer built from scratch."""

    def __init__(self, max_features=None, min_df=1, stop_words=None):
        self.max_features = max_features   # keep only the N most frequent words
        self.min_df = min_df               # ignore words appearing in fewer than min_df docs
        self.stop_words = set(stop_words) if stop_words else set()
        self.vocabulary_ = {}              # word → column index mapping

    def _tokenize(self, text):
        """Lowercase, remove non-alpha characters, split on whitespace."""
        text = re.sub(r'[^a-z\s]', '', text.lower())   # keep only letters and spaces
        tokens = text.split()                            # split on any whitespace
        return [t for t in tokens if t not in self.stop_words]  # remove stop words

    def fit(self, documents):
        """Learn vocabulary from a list of documents."""
        doc_freq = Counter()                             # how many docs contain each word
        for doc in documents:
            unique_words = set(self._tokenize(doc))      # unique words per doc
            doc_freq.update(unique_words)                # increment doc count for each

        # Keep only words that appear in at least min_df documents
        vocab = {w: f for w, f in doc_freq.items() if f >= self.min_df}

        # Sort by descending frequency; truncate to max_features if set
        sorted_vocab = sorted(vocab.items(), key=lambda x: -x[1])
        if self.max_features:
            sorted_vocab = sorted_vocab[:self.max_features]

        # Assign each word a column index
        self.vocabulary_ = {w: i for i, (w, _) in enumerate(sorted_vocab)}
        return self

    def transform(self, documents):
        """Convert documents to a count matrix (n_docs x n_vocab)."""
        X = np.zeros((len(documents), len(self.vocabulary_)))  # start with all zeros
        for i, doc in enumerate(documents):
            for word in self._tokenize(doc):
                if word in self.vocabulary_:              # ignore out-of-vocab words
                    X[i, self.vocabulary_[word]] += 1    # increment the word's column
        return X

    def fit_transform(self, documents):
        return self.fit(documents).transform(documents)


# Apply to listing descriptions
cv_scratch = CountVectorizerScratch(max_features=50, min_df=2)
X_count = cv_scratch.fit_transform(df['description'].tolist())

print(f"Count matrix shape: {X_count.shape}  (200 listings × up to 50 words)")
print(f"Sparsity: {(X_count == 0).mean():.1%} of cells are zero")

# Show top-20 most common words
word_totals = X_count.sum(axis=0)                        # sum across all documents
idx_sorted  = np.argsort(word_totals)[::-1]              # descending order
inv_vocab   = {v: k for k, v in cv_scratch.vocabulary_.items()}  # index → word

top20_words  = [inv_vocab[i] for i in idx_sorted[:20]]
top20_counts = [word_totals[i] for i in idx_sorted[:20]]

print("\nTop-20 words (scratch):")
for w, c in zip(top20_words, top20_counts):
    print(f"  {w:<20} {int(c):>4} occurrences")

In [ ]:
np.random.seed(42)

# ── Verify scratch CountVectorizer against sklearn ─────────────────────────
# sklearn uses a slightly different tokeniser, so we compare overall structure
cv_sklearn = CountVectorizer(max_features=50, min_df=2)
X_sklearn  = cv_sklearn.fit_transform(df['description'].tolist()).toarray()

# Find words present in BOTH vocabularies and compare their column sums
common_words = set(cv_scratch.vocabulary_) & set(cv_sklearn.vocabulary_)
print(f"Words in scratch vocab : {len(cv_scratch.vocabulary_)}")
print(f"Words in sklearn vocab : {len(cv_sklearn.vocabulary_)}")
print(f"Words in both          : {len(common_words)}")

scratch_counts = {w: X_count[:, cv_scratch.vocabulary_[w]].sum() for w in common_words}
sklearn_counts  = {w: X_sklearn[:, cv_sklearn.vocabulary_[w]].sum() for w in common_words}

match = sum(scratch_counts[w] == sklearn_counts[w] for w in common_words)
print(f"\nExact token-count match for {match}/{len(common_words)} shared words")
print("(Minor differences are expected due to tokeniser variations)")

# Visualise top-20 scratch words
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top20_words[::-1], top20_counts[::-1], color='steelblue')  # horizontal bars
ax.set_xlabel('Total occurrences across all listings')
ax.set_title('Top-20 Most Frequent Words (CountVectorizer from scratch)')
plt.tight_layout()
plt.show()

## Stop Words: Removing the Noise

Words like **"the"**, **"a"**, **"is"**, **"and"**, **"for"** appear in almost every document. They carry no discriminative signal — knowing a listing contains "the" tells you nothing useful.

These are called **stop words**. Removing them:
- Shrinks the vocabulary (faster training, less memory)
- Reduces noise in the feature matrix
- Lets the model focus on content words

**Caution:** In some tasks (sentiment analysis, negation detection), stop words like "not" are critical. Always check whether removal helps or hurts on your specific task.


In [ ]:
np.random.seed(42)

STOP_WORDS = {
    'a', 'an', 'the', 'and', 'or', 'but', 'in', 'on', 'at', 'to',
    'for', 'of', 'with', 'is', 'are', 'was', 'be', 'by', 'from',
    'it', 'its', 'this', 'that', 'as', 'up', 'out', 'into', 'near',
}

# Compare top-20 words WITH and WITHOUT stop word removal
cv_no_sw  = CountVectorizerScratch(max_features=50, min_df=2)
cv_with_sw = CountVectorizerScratch(max_features=50, min_df=2, stop_words=list(STOP_WORDS))

X_no_sw   = cv_no_sw.fit_transform(df['description'].tolist())
X_with_sw = cv_with_sw.fit_transform(df['description'].tolist())

def top_words(X, vocab, n=10):
    inv = {v: k for k, v in vocab.items()}
    totals = X.sum(axis=0)
    idx = np.argsort(totals)[::-1][:n]
    return [(inv[i], int(totals[i])) for i in idx]

print("Top-10 WITHOUT stop word removal:")
for w, c in top_words(X_no_sw, cv_no_sw.vocabulary_):
    print(f"  {w:<20} {c:>4}")

print("\nTop-10 WITH stop word removal:")
for w, c in top_words(X_with_sw, cv_with_sw.vocabulary_):
    print(f"  {w:<20} {c:>4}")

# Compare Ridge R² with and without stop word removal
y = df['price'].values
X_train_ns, X_test_ns, y_train, y_test = train_test_split(X_no_sw,   y, test_size=0.2, random_state=42)
X_train_ws, X_test_ws, _,      _      = train_test_split(X_with_sw,  y, test_size=0.2, random_state=42)

r2_no_sw   = r2_score(y_test, Ridge().fit(X_train_ns, y_train).predict(X_test_ns))
r2_with_sw = r2_score(y_test, Ridge().fit(X_train_ws, y_train).predict(X_test_ws))

print(f"\nRidge R² WITHOUT stop words: {r2_no_sw:.3f}")
print(f"Ridge R² WITH stop word removal: {r2_with_sw:.3f}")
print("Stop word removal typically helps — fewer noise columns for the model to fit.")

## TF-IDF: Rewarding Distinctive Words

Plain word counts have a problem: long documents will have higher counts for every word. TF-IDF fixes this and adds the *distinctiveness* dimension.

**Two components:**

**Term Frequency (TF):** How often does word *t* appear in document *d*, relative to document length?
$$\text{TF}(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total words in } d}$$

**Inverse Document Frequency (IDF):** How rare is word *t* across all N documents?
$$\text{IDF}(t) = \log\left(\frac{N}{\text{df}(t)}\right)$$
where df(t) = number of documents containing *t*.

**TF-IDF score:**
$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

A word that is frequent in *this* document but rare across *all* documents gets a high score — it's the listing's unique signature.


In [ ]:
np.random.seed(42)

class TFIDFVectorizerScratch(CountVectorizerScratch):
    """TF-IDF vectorizer built on top of CountVectorizerScratch."""

    def fit(self, documents):
        super().fit(documents)                            # build the vocabulary
        n = len(documents)                               # total number of documents

        # Count document frequency for every word in the vocabulary
        doc_freq = np.zeros(len(self.vocabulary_))
        for doc in documents:
            words = set(self._tokenize(doc))             # unique words in this doc
            for w in words:
                if w in self.vocabulary_:
                    doc_freq[self.vocabulary_[w]] += 1   # increment df counter

        # Smoothed IDF (sklearn formula): avoids division by zero
        # +1 in numerator and denominator prevents log(0); final +1 keeps IDF >= 1
        self.idf_ = np.log((n + 1) / (doc_freq + 1)) + 1
        return self

    def transform(self, documents):
        tf = super().transform(documents)                # raw counts from parent

        # Normalise counts by document length → term frequency
        row_sums = tf.sum(axis=1, keepdims=True)         # total words per document
        tf = tf / (row_sums + 1e-10)                     # add epsilon to avoid /0

        tfidf = tf * self.idf_                           # multiply TF by IDF weights

        # L2-normalise each document vector (so document length doesn't dominate)
        norms = np.linalg.norm(tfidf, axis=1, keepdims=True)
        return tfidf / (norms + 1e-10)


# Fit on descriptions, show IDF values for selected words
tfidf_scratch = TFIDFVectorizerScratch(max_features=100, min_df=1, stop_words=list(STOP_WORDS))
X_tfidf = tfidf_scratch.fit_transform(df['description'].tolist())

# Show IDF for interesting words
interesting_words = ['cozy', 'luxury', 'budget', 'family', 'kitchen', 'garden', 'wifi', 'studio']
print("IDF values (higher = rarer = more distinctive):")
for w in interesting_words:
    if w in tfidf_scratch.vocabulary_:
        idf_val = tfidf_scratch.idf_[tfidf_scratch.vocabulary_[w]]
        print(f"  {w:<20} IDF = {idf_val:.3f}")
    else:
        print(f"  {w:<20} not in vocabulary")

In [ ]:
np.random.seed(42)

# ── Show that 'luxury' keywords have high TF-IDF for luxury listings ───────
listing_type_arr = df['listing_type'].values

# For each listing type, compute mean TF-IDF across its listings
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax_idx, ltype in enumerate(['luxury', 'cozy', 'budget', 'family']):
    mask  = listing_type_arr == ltype                    # boolean mask for this type
    mean_tfidf = X_tfidf[mask].mean(axis=0)             # average TF-IDF per word

    inv_vocab  = {v: k for k, v in tfidf_scratch.vocabulary_.items()}
    top_idx    = np.argsort(mean_tfidf)[::-1][:10]      # top-10 words for this type
    top_words_t = [inv_vocab[i] for i in top_idx]
    top_scores  = [mean_tfidf[i] for i in top_idx]

    color_map = {'luxury': 'gold', 'cozy': 'coral', 'budget': 'mediumpurple', 'family': 'mediumseagreen'}
    axes[ax_idx].barh(top_words_t[::-1], top_scores[::-1], color=color_map[ltype])
    axes[ax_idx].set_title(f'Top TF-IDF Words — {ltype.capitalize()} listings')
    axes[ax_idx].set_xlabel('Mean TF-IDF score')

plt.suptitle('Characteristic Words per Listing Type (TF-IDF)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

# ── Verify TF-IDF scratch against sklearn ──────────────────────────────────
tfidf_sklearn = TfidfVectorizer(
    max_features=100,
    min_df=1,
    stop_words=None,        # we apply stop words manually in scratch
    smooth_idf=True,
    sublinear_tf=False,
    norm='l2'
)

# Re-fit scratch without custom stop words to match sklearn defaults
tfidf_scratch_plain = TFIDFVectorizerScratch(max_features=100, min_df=1)
X_scratch_plain = tfidf_scratch_plain.fit_transform(df['description'].tolist())
X_sklearn_tfidf = tfidf_sklearn.fit_transform(df['description'].tolist()).toarray()

# For words in both vocabularies, compute cosine similarity between the two representations
common = set(tfidf_scratch_plain.vocabulary_) & set(tfidf_sklearn.vocabulary_)

scratch_cols = np.array([tfidf_scratch_plain.vocabulary_[w] for w in common])
sklearn_cols  = np.array([tfidf_sklearn.vocabulary_[w]       for w in common])

A = X_scratch_plain[:, scratch_cols]  # scratch matrix for common words
B = X_sklearn_tfidf[:, sklearn_cols]  # sklearn matrix for common words

# Row-wise cosine similarity: dot(a,b) / (|a| * |b|)
dot    = (A * B).sum(axis=1)                             # element-wise multiply then sum
norm_a = np.linalg.norm(A, axis=1)
norm_b = np.linalg.norm(B, axis=1)
cos_sim = dot / (norm_a * norm_b + 1e-10)

print(f"Words compared: {len(common)}")
print(f"Mean cosine similarity (scratch vs sklearn): {cos_sim.mean():.4f}")
print(f"Min cosine similarity: {cos_sim.min():.4f}")
print(f"Fraction of rows with cosine similarity > 0.99: {(cos_sim > 0.99).mean():.1%}")
print("\nHigh cosine similarity confirms our scratch implementation matches sklearn closely.")

## N-grams: Capturing Phrases, Not Just Words

**Unigrams** treat each word independently: ["chef", "kitchen"]. The problem is that "chef" alone tells you less than "chef kitchen" together.

**N-grams** capture sequences of N consecutive words:
- **Unigram (1-gram):** "chef", "kitchen", "panoramic", "views"
- **Bigram (2-gram):** "chef kitchen", "panoramic views", "near centre"
- **Trigram (3-gram):** "chef kitchen and", "panoramic city views"

**When bigrams help:**
- Phrases with specific meaning: "instant book", "shared bathroom", "city centre"
- Negations: "not clean" vs. "not" + "clean" separately
- Compound adjectives: "budget friendly" vs. "budget" alone

**Trade-off:** Bigrams dramatically expand vocabulary size. Use `max_features` to control it.


In [ ]:
np.random.seed(42)

# ── Unigrams vs. Unigrams+Bigrams: Ridge R² comparison ─────────────────────
y = df['price'].values

# Unigrams only
tfidf_uni = TfidfVectorizer(ngram_range=(1, 1), max_features=100, min_df=1)
X_uni     = tfidf_uni.fit_transform(df['description'].tolist())

# Unigrams + Bigrams
tfidf_bi = TfidfVectorizer(ngram_range=(1, 2), max_features=200, min_df=1)
X_bi     = tfidf_bi.fit_transform(df['description'].tolist())

# Split data (same indices for fair comparison)
idx = np.arange(len(y))
train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=42)

r2_uni = r2_score(
    y[test_idx],
    Ridge(alpha=1.0).fit(X_uni[train_idx], y[train_idx]).predict(X_uni[test_idx])
)
r2_bi = r2_score(
    y[test_idx],
    Ridge(alpha=1.0).fit(X_bi[train_idx], y[train_idx]).predict(X_bi[test_idx])
)

print(f"Ridge R² — unigrams only       : {r2_uni:.4f}")
print(f"Ridge R² — unigrams + bigrams  : {r2_bi:.4f}")
print(f"Improvement from bigrams: {r2_bi - r2_uni:+.4f}")

# Show top bigrams by TF-IDF weight
feature_names = tfidf_bi.get_feature_names_out()
bigrams_only  = [(f, i) for i, f in enumerate(feature_names) if ' ' in f]  # bigrams contain a space

# Mean TF-IDF across all docs
mean_tfidf_bi = X_bi.toarray().mean(axis=0)
bigram_scores  = sorted([(f, mean_tfidf_bi[i]) for f, i in bigrams_only], key=lambda x: -x[1])[:10]

print("\nTop-10 bigrams by mean TF-IDF:")
for phrase, score in bigram_scores:
    print(f"  '{phrase:<30}' score = {score:.4f}")

## Other Text Feature Types

Beyond word n-grams, there are other ways to featurise text:

| Analyser | What it captures | Good for |
|---|---|---|
| Word unigrams | Word presence/frequency | Most tasks |
| Word bigrams | Common phrases | When phrases matter |
| **Char n-grams** | Sub-word patterns, morphology | Noisy text, misspellings, morphologically rich languages |
| Subword (BPE) | Efficient vocab for rare words | Large language models |
| Sentence embeddings | Semantic meaning | Semantic search, paraphrase detection |

**Character n-grams** are particularly useful when:
- Your text contains typos: "luxurious" and "luxurios" share many character n-grams
- Morphology matters: "clean", "cleaner", "cleanest" share the stem "clean"
- Languages have complex morphology (German, Finnish, Arabic)

*(Sentence embeddings from transformer models are covered in the NLP module.)*


In [ ]:
np.random.seed(42)

# ── Character n-gram demo ──────────────────────────────────────────────────
# analyzer='char_wb' pads each word with spaces before extracting char n-grams
char_vectorizer = TfidfVectorizer(
    analyzer='char_wb',    # character n-grams, word boundaries respected
    ngram_range=(3, 5),    # trigrams to 5-grams
    max_features=50
)

X_char = char_vectorizer.fit_transform(df['description'].tolist())
char_features = char_vectorizer.get_feature_names_out()

print("Sample character n-grams learned:")
# Show n-grams that relate to 'cozy'
cozy_related = [f for f in char_features if 'coz' in f or 'ozy' in f]
luxury_related = [f for f in char_features if 'lux' in f or 'ury' in f]

print(f"  Cozy-related char n-grams: {cozy_related}")
print(f"  Luxury-related char n-grams: {luxury_related}")

# Demonstrate morphology capture: 'cozy' and variations
test_words = ['cozy', 'cozy ', ' cozy', 'coziness', 'cozy studio']  # intentional variations
test_vecs  = char_vectorizer.transform(test_words).toarray()

# Cosine similarity between 'cozy' and its variants
base = test_vecs[0]
print("\nCosine similarity to 'cozy':")
for word, vec in zip(test_words[1:], test_vecs[1:]):
    sim = np.dot(base, vec) / (np.linalg.norm(base) * np.linalg.norm(vec) + 1e-10)
    print(f"  '{word}'  →  cosine sim = {sim:.4f}")

print("\nCharacter n-grams group morphologically similar words together.")

In [ ]:
np.random.seed(42)

# ── Combine text + numeric features ───────────────────────────────────────
# Text-only TF-IDF matrix (sparse)
tfidf_combo = TfidfVectorizer(ngram_range=(1, 2), max_features=150, min_df=1)
X_text      = tfidf_combo.fit_transform(df['description'].tolist())  # sparse: (200, 150)

# Numeric features as a sparse matrix so we can hstack
X_numeric_dense  = df[['bedrooms', 'review_score']].values.astype(float)  # (200, 2)
X_numeric_sparse = scipy.sparse.csr_matrix(X_numeric_dense)               # convert to sparse

# Horizontal stack: combine text and numeric columns
X_combined = scipy.sparse.hstack([X_text, X_numeric_sparse])              # (200, 152)

print(f"Text features shape    : {X_text.shape}")
print(f"Numeric features shape : {X_numeric_dense.shape}")
print(f"Combined features shape: {X_combined.shape}")

# Evaluate three models with the same train/test split
y = df['price'].values
train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42)

results = {}
for name, X in [('Numeric only', X_numeric_sparse),
                ('Text only',    X_text),
                ('Text + Numeric', X_combined)]:
    model = Ridge(alpha=1.0)
    model.fit(X[train_idx], y[train_idx])              # train
    preds = model.predict(X[test_idx])                 # predict
    results[name] = r2_score(y[test_idx], preds)

print("\nRidge R² comparison:")
for name, r2 in results.items():
    bar = '█' * int(max(0, r2) * 40)                  # visual bar
    print(f"  {name:<20} R² = {r2:.4f}  {bar}")

# Plot R² comparison
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(list(results.keys()), list(results.values()), color=['#4C72B0', '#DD8452', '#55A868'])
ax.set_xlabel('R² (higher is better)')
ax.set_title('Text + Numeric features beat either alone')
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

# ── Feature importance from Ridge coefficients ─────────────────────────────
# Fit Ridge on text-only features to interpret word coefficients
model_text = Ridge(alpha=1.0)
model_text.fit(X_text, y)                              # fit on all data for interpretation

feature_names_combo = tfidf_combo.get_feature_names_out()
coefs               = model_text.coef_                 # one coefficient per text feature

# Top 10 words driving price UP (highest positive coefficients)
top_pos_idx = np.argsort(coefs)[::-1][:10]
# Top 10 words driving price DOWN (most negative coefficients)
top_neg_idx = np.argsort(coefs)[:10]

print("Words that INCREASE predicted price (positive coefficients):")
for i in top_pos_idx:
    print(f"  '{feature_names_combo[i]:<30}'  coef = +{coefs[i]:.2f}")

print("\nWords that DECREASE predicted price (negative coefficients):")
for i in top_neg_idx:
    print(f"  '{feature_names_combo[i]:<30}'  coef = {coefs[i]:.2f}")

# Visualise
top20_idx   = np.concatenate([top_pos_idx[:10], top_neg_idx[:10]])
top20_names = [feature_names_combo[i] for i in top20_idx]
top20_coefs = coefs[top20_idx]
colors      = ['green' if c > 0 else 'red' for c in top20_coefs]

sort_order  = np.argsort(top20_coefs)                  # sort for readable chart

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(
    [top20_names[i] for i in sort_order],
    [top20_coefs[i] for i in sort_order],
    color=[colors[i] for i in sort_order]
)
ax.axvline(0, color='black', linewidth=0.8)            # zero line
ax.set_xlabel('Ridge coefficient (impact on predicted price £/night)')
ax.set_title('Top-10 Price-Increasing and Price-Decreasing Words')
plt.tight_layout()
plt.show()

## Self-Check Questions

**Q1.** A word appears in 190 out of 200 documents. Will it have a high or low IDF score? Why?

<details><summary>Show answer</summary>

**Low IDF.** IDF = log(N / df(t)) = log(200 / 190) ≈ log(1.05) ≈ 0.05. The word is nearly universal — it's not distinctive. A word in only 2 documents would have IDF = log(200/2) = log(100) ≈ 4.6, much higher.

</details>

---

**Q2.** Why does adding bigrams usually improve model performance for property description tasks?

<details><summary>Show answer</summary>

Because real-estate language relies on **compound phrases**: "chef kitchen" means something different from "chef" + "kitchen" independently. "Instant book", "shared bathroom", "city centre", "panoramic views" — these are meaningful units that bigrams capture and unigrams miss. The spatial co-occurrence of words encodes semantic information that single-word models cannot represent.

</details>

---

**Q3.** You build a TF-IDF model and find the word "apartment" has the highest coefficient in your Ridge model. Is this a useful feature? What might you do about it?

<details><summary>Show answer</summary>

Probably **not very useful** — "apartment" is likely in most listings (high df → low IDF), so it may have picked up a spurious correlation. You should: (1) check its IDF score to confirm it's not distinctive, (2) add it to your stop word list if it's nearly universal, (3) check if it's a proxy for listing type (apartments vs. rooms) — if so, the listing type variable captures this better as a categorical feature.

</details>


## Key Takeaways

1. **TF-IDF is the standard text baseline.** Before trying transformers or embeddings, TF-IDF with bigrams often gets you 80% of the way there, in seconds.

2. **Stop word removal reduces noise.** Remove generic words that appear everywhere — they add columns but no signal.

3. **Bigrams help with phrase-dependent domains.** Property descriptions, medical notes, and legal documents all rely on compound phrases. Include bigrams.

4. **Always combine text with numeric features.** `scipy.sparse.hstack` makes this easy. Text + numeric almost always beats either alone.

5. **Ridge coefficients give interpretable word importance.** Positive coefficients = price-increasing words. Negative = price-decreasing. This is a free sanity check on your model.

6. **Character n-grams handle noisy text.** If your text has typos, abbreviations, or morphological variation, `analyzer='char_wb'` is worth trying.
